# Module 7: Advanced Data Modeling & Business Analytics

**For**: Senior Data Engineers with Data Modeling focus

## Topics
1. Star Schema Implementation in Spark
2. Fact Table Design Patterns
3. Business KPI & Metrics Calculation
4. Data Vault Concepts (Hub, Link, Satellite)
5. Incremental Load Patterns

In [ ]:
# ── SparkSession: Databricks Connect (remote) / Local fallback ──
from pathlib import Path

try:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.serverless().getOrCreate()
    MODE = 'databricks'
    S3_RAW = "s3a://sparkling-data-test/data/raw"
    print(f"✅ Databricks Connect | Spark {spark.version}")
except Exception:
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.appName("Module07-DataModeling").master("local[*]").config("spark.sql.shuffle.partitions", "8").getOrCreate()
    MODE = 'local'
    S3_RAW = None
    print(f"✅ Local Spark {spark.version} | UI: http://localhost:4040")

DATA_RAW = Path("../data/raw")  # local CSV fallback path
print(f"Mode: {MODE}")

In [ ]:
# ── Load data (S3 Parquet or local CSV) ──
customers = spark.read.parquet(f"{S3_RAW}/customers") if MODE == "databricks" else spark.read.csv(str(DATA_RAW / "customers.csv"), header=True, inferSchema=True)
accounts = spark.read.parquet(f"{S3_RAW}/accounts") if MODE == "databricks" else spark.read.csv(str(DATA_RAW / "accounts.csv"), header=True, inferSchema=True)
transactions = spark.read.parquet(f"{S3_RAW}/transactions") if MODE == "databricks" else spark.read.csv(str(DATA_RAW / "transactions.csv"), header=True, inferSchema=True)
branches = spark.read.parquet(f"{S3_RAW}/branches") if MODE == "databricks" else spark.read.csv(str(DATA_RAW / "branches.csv"), header=True, inferSchema=True)


---
## 1. Star Schema: Fact & Dimension Tables

### Design Philosophy
- **Fact Table**: Measures (amounts, counts) + foreign keys to dimensions
- **Dimensions**: Descriptive attributes for slicing/dicing
- **Conformed Dimensions**: Shared across multiple facts (e.g., dim_date)

In [ ]:
# === DIM_DATE (Conformed Dimension) ===
from datetime import datetime, timedelta

dates = [(datetime(2025, 1, 1) + timedelta(days=i)).strftime("%Y-%m-%d") for i in range(365)]
dim_date = spark.createDataFrame([(d,) for d in dates], ["date_id"]).withColumn(
    "date_key", monotonically_increasing_id()
).withColumn("year", year(to_date(col("date_id")))
).withColumn("month", month(to_date(col("date_id")))
).withColumn("quarter", quarter(to_date(col("date_id")))
).withColumn("day_of_week", dayofweek(to_date(col("date_id")))
).withColumn("is_weekend", col("day_of_week").isin([1, 7]))

dim_date.show(5)

In [ ]:
# === DIM_CUSTOMER (with surrogate key) ===
dim_customer = customers.select(
    monotonically_increasing_id().alias("customer_key"),
    col("customer_id"),
    col("name"),
    col("segment"),
    col("kyc_status"),
    # Derived attributes for analysis
    when(col("segment").isin(["HNW", "UHNW"]), "High Value").otherwise("Standard").alias("customer_tier")
)
dim_customer.show(5)

In [ ]:
# === DIM_BRANCH (Role-playing dim: can be 'originating_branch', 'servicing_branch') ===
dim_branch = branches.select(
    monotonically_increasing_id().alias("branch_key"),
    col("branch_id"),
    col("branch_name"),
    col("region"),
    # Derived: Region grouping for reporting
    when(col("region").isin(["Hanoi", "Ho Chi Minh"]), "Major City").otherwise("Province").alias("region_type")
)
dim_branch.show(5)

In [ ]:
# === FACT_TRANSACTION (Transactional grain: 1 row per transaction) ===
# Join to get surrogate keys from dimensions
txn_enriched = transactions.withColumn("txn_date", to_date(col("txn_datetime")))
txn_with_account = txn_enriched.join(accounts.select("account_id", "customer_id", "branch_id"), "account_id")

fact_transaction = txn_with_account.join(dim_customer.select("customer_key", "customer_id"), "customer_id"
).join(dim_branch.select("branch_key", "branch_id"), "branch_id"
).join(dim_date.select("date_key", col("date_id").alias("txn_date")), "txn_date"
).select(
    # Degenerate dimension (transaction ID kept in fact)
    col("txn_id"),
    # Foreign keys to dimensions
    col("date_key"),
    col("customer_key"),
    col("branch_key"),
    # Measures
    col("amount"),
    when(col("txn_type").isin(["Deposit", "Transfer In", "Interest"]), col("amount")).otherwise(0).alias("credit_amount"),
    when(col("txn_type").isin(["Withdrawal", "Transfer Out", "Payment", "Fee"]), col("amount")).otherwise(0).alias("debit_amount"),
    lit(1).alias("txn_count"),
    # Attributes for filtering
    col("txn_type"),
    col("channel"),
    col("status")
)
fact_transaction.show(5)

---
## 2. Business KPIs & Metrics

### Common Banking KPIs

In [ ]:
# === KPI 1: Customer Lifetime Value (CLV) Proxy ===
# Total transaction volume per customer as CLV proxy
clv_metric = fact_transaction.groupBy("customer_key").agg(
    sum("amount").alias("total_volume"),
    count("txn_id").alias("txn_count"),
    (sum("credit_amount") - sum("debit_amount")).alias("net_flow")
).join(dim_customer, "customer_key")

clv_metric.orderBy(col("total_volume").desc()).show(10)

In [ ]:
# === KPI 2: Monthly Active Customers (MAU) ===
mau = fact_transaction.join(dim_date, "date_key").groupBy("year", "month").agg(
    countDistinct("customer_key").alias("mau"),
    sum("amount").alias("monthly_volume")
).orderBy("year", "month")
mau.show(12)

In [ ]:
# === KPI 3: Channel Mix Analysis ===
channel_mix = fact_transaction.groupBy("channel").agg(
    sum("txn_count").alias("total_txns"),
    sum("amount").alias("total_amount")
).withColumn(
    "txn_share", round(col("total_txns") / sum("total_txns").over(Window.partitionBy()) * 100, 2)
).withColumn(
    "amount_share", round(col("total_amount") / sum("total_amount").over(Window.partitionBy()) * 100, 2)
).orderBy(col("total_amount").desc())

channel_mix.show()

In [ ]:
# === KPI 4: Customer Segment Performance ===
segment_perf = fact_transaction.join(dim_customer, "customer_key").groupBy("segment", "customer_tier").agg(
    countDistinct("customer_key").alias("customer_count"),
    sum("amount").alias("total_volume"),
    round(avg("amount"), 0).alias("avg_txn_amount")
).withColumn(
    "volume_per_customer", round(col("total_volume") / col("customer_count"), 0)
).orderBy(col("volume_per_customer").desc())

segment_perf.show()

---
## 3. Aggregate Fact Tables (Periodic Snapshot)

Pre-aggregated facts for faster BI queries.

In [ ]:
# === Monthly Customer Snapshot (Periodic Snapshot Grain) ===
monthly_snapshot = fact_transaction.join(dim_date, "date_key").groupBy(
    "year", "month", "customer_key"
).agg(
    sum("txn_count").alias("monthly_txn_count"),
    sum("amount").alias("monthly_volume"),
    sum("credit_amount").alias("monthly_credits"),
    sum("debit_amount").alias("monthly_debits"),
    countDistinct("channel").alias("channels_used")
)

monthly_snapshot.show(10)

---
## 4. Data Vault Concepts (Optional Advanced)

### Core Entities
- **Hub**: Business keys (customer_id, account_id)
- **Link**: Relationships between hubs
- **Satellite**: Descriptive attributes with history

In [ ]:
# === HUB_CUSTOMER ===
hub_customer = customers.select(
    md5(col("customer_id")).alias("hub_customer_hk"),  # Hash key
    col("customer_id"),
    current_timestamp().alias("load_dts"),
    lit("SOURCE_SYSTEM").alias("record_source")
)
hub_customer.show(3)

# === SAT_CUSTOMER (Satellite with history) ===
sat_customer = customers.select(
    md5(col("customer_id")).alias("hub_customer_hk"),
    md5(concat_ws("|", col("name"), col("segment"), col("kyc_status"))).alias("hash_diff"),
    col("name"), col("segment"), col("kyc_status"),
    current_timestamp().alias("load_dts")
)
sat_customer.show(3)

---
## 5. Incremental Load Pattern

In [ ]:
def incremental_load(source_df, target_path, key_col, watermark_col, mode="append"):
    """
    Pattern for incremental loads based on watermark/timestamp.
    
    In production:
    1. Read last watermark from control table
    2. Filter source for records > watermark
    3. Transform and write
    4. Update control table with new watermark
    """
    # Simulate reading last watermark (normally from control table)
    last_watermark = "2025-06-01 00:00:00"
    
    # Filter incremental records
    incremental = source_df.filter(col(watermark_col) > last_watermark)
    
    print(f"Incremental records: {incremental.count()}")
    return incremental

# Example: Load only new transactions
new_txns = incremental_load(transactions, "path/to/target", "txn_id", "txn_datetime")

---
## Practice: Business Scenario

**Scenario**: Finance team needs a monthly report showing:
1. Revenue by customer segment
2. Top 10 customers by volume
3. Channel trend over last 6 months
4. Regional breakdown

In [ ]:
# Your solution here:


In [ ]:
spark.stop()